# 18n — June 2026 Hong Kong Polymarket contract-event audit

This notebook audits all 30 June 2026 dates before any date is admitted to the expanded HKO–Polymarket sample.

It discovers and archives the official Polymarket Gamma event payloads, parses the event labels, checks the HKO Daily Extract one-decimal settlement family, verifies the 11-contract partition, extracts CLOB token identifiers and writes canonical audit, issue, report and integrity outputs.

It does **not** create a branch, commit, push, pull request, reminder or notification.

In [1]:
from __future__ import annotations

import hashlib
import json
import platform
import re
import sys
import time
from datetime import date, datetime, timezone
from pathlib import Path
from typing import Any

import pandas as pd
import requests
from IPython.display import display

REPO_ROOT = Path.cwd().resolve()
if not (REPO_ROOT / '.git').exists():
    raise RuntimeError(
        'Run this notebook from the repository root. '
        f'Current directory: {REPO_ROOT}'
    )

STEP = '18n'
MONTH_DATES = [date(2026, 6, day) for day in range(1, 31)]
RAW_DIR = REPO_ROOT / 'data/raw/18n_june_2026_contract_event_audit'
OUT_DIR = REPO_ROOT / 'data/processed/18n_june_2026_contract_event_audit'
REPORT_DIR = REPO_ROOT / 'reports/18n_june_2026_contract_event_audit'
for directory in (RAW_DIR, OUT_DIR, REPORT_DIR):
    directory.mkdir(parents=True, exist_ok=True)

BASE_URL = 'https://gamma-api.polymarket.com'
USER_AGENT = '2026MScWeatherForecastingPolymarket/18n-june-contract-audit'
REQUEST_TIMEOUT_SECONDS = 60
REQUEST_SLEEP_SECONDS = 0.10

session = requests.Session()
session.headers.update({'User-Agent': USER_AGENT, 'Accept': 'application/json'})


def scalar_text(value: Any) -> str:
    if value is None:
        return ''
    if isinstance(value, (dict, list, tuple)):
        return json.dumps(value, ensure_ascii=False, sort_keys=True)
    return str(value)


def parse_jsonish_list(value: Any) -> list[Any]:
    if value is None:
        return []
    if isinstance(value, list):
        return value
    if isinstance(value, tuple):
        return list(value)
    if isinstance(value, str):
        text = value.strip()
        if not text:
            return []
        try:
            parsed = json.loads(text)
            return parsed if isinstance(parsed, list) else [parsed]
        except json.JSONDecodeError:
            return [part.strip() for part in text.split(',') if part.strip()]
    return [value]


def ordinal(day: int) -> str:
    if 10 <= day % 100 <= 20:
        suffix = 'th'
    else:
        suffix = {1: 'st', 2: 'nd', 3: 'rd'}.get(day % 10, 'th')
    return f'{day}{suffix}'


def candidate_slugs(event_date: date) -> list[str]:
    day = event_date.day
    return [
        f'highest-temperature-in-hong-kong-on-june-{day}-2026',
        f'highest-temperature-in-hong-kong-on-june-{ordinal(day)}-2026',
        f'highest-temperature-in-hong-kong-on-june-{day}',
        f'highest-temperature-in-hong-kong-june-{day}-2026',
    ]


def request_json(
    url: str,
    *,
    params: dict[str, Any] | None = None,
    attempts: int = 4,
) -> tuple[Any | None, dict[str, Any]]:
    last_error = ''
    for attempt in range(1, attempts + 1):
        try:
            response = session.get(url, params=params, timeout=REQUEST_TIMEOUT_SECONDS)
            metadata = {
                'request_url': response.url,
                'http_status': response.status_code,
                'attempt': attempt,
            }
            if response.status_code == 404:
                return None, {**metadata, 'error': 'HTTP 404'}
            if response.status_code == 429 or response.status_code >= 500:
                last_error = f'HTTP {response.status_code}'
                time.sleep(1.5 * attempt)
                continue
            response.raise_for_status()
            return response.json(), {**metadata, 'error': ''}
        except (requests.RequestException, ValueError) as exc:
            last_error = f'{type(exc).__name__}: {exc}'
            time.sleep(1.5 * attempt)
    return None, {
        'request_url': url,
        'http_status': None,
        'attempt': attempts,
        'error': last_error or 'Unknown request failure',
    }


def title_date_match(event: dict[str, Any], event_date: date) -> bool:
    text = ' '.join([
        scalar_text(event.get('title')),
        scalar_text(event.get('slug')),
        scalar_text(event.get('description')),
    ]).lower()
    if not all(term in text for term in ['hong kong', 'temperature', 'june', '2026']):
        return False
    day = event_date.day
    patterns = [
        rf'\bjune[\s\-]+{day}\b',
        rf'\bjune[\s\-]+{ordinal(day)}\b',
        rf'\b{day}[\s\-]+june\b',
        rf'\b{ordinal(day)}[\s\-]+june\b',
    ]
    return any(re.search(pattern, text, flags=re.IGNORECASE) for pattern in patterns)


def fetch_event_by_slug(slug: str) -> tuple[dict[str, Any] | None, dict[str, Any]]:
    payload, metadata = request_json(f'{BASE_URL}/events/slug/{slug}')
    if isinstance(payload, dict) and payload:
        return payload, metadata
    return None, metadata


def discover_event(event_date: date) -> tuple[dict[str, Any] | None, list[dict[str, Any]]]:
    request_log: list[dict[str, Any]] = []
    for slug in candidate_slugs(event_date):
        event, metadata = fetch_event_by_slug(slug)
        request_log.append({
            'event_date': event_date.isoformat(),
            'method': 'event_by_slug',
            'candidate_slug': slug,
            **metadata,
        })
        if event is not None and title_date_match(event, event_date):
            return event, request_log

    query = f'highest temperature in Hong Kong on June {event_date.day} 2026'
    payload, metadata = request_json(
        f'{BASE_URL}/public-search',
        params={
            'q': query,
            'events_status': 'closed',
            'limit_per_type': 30,
            'page': 1,
            'keep_closed_markets': 1,
            'search_tags': 'false',
            'search_profiles': 'false',
        },
    )
    request_log.append({
        'event_date': event_date.isoformat(),
        'method': 'public_search',
        'candidate_slug': '',
        **metadata,
    })
    events = payload.get('events', []) if isinstance(payload, dict) else []
    matched = [
        event for event in events
        if isinstance(event, dict) and title_date_match(event, event_date)
    ]
    matched.sort(key=lambda event: (
        'hong-kong' not in scalar_text(event.get('slug')).lower(),
        'highest-temperature' not in scalar_text(event.get('slug')).lower(),
        scalar_text(event.get('slug')),
    ))
    for search_event in matched:
        slug = scalar_text(search_event.get('slug'))
        if not slug:
            continue
        event, metadata = fetch_event_by_slug(slug)
        request_log.append({
            'event_date': event_date.isoformat(),
            'method': 'search_result_by_slug',
            'candidate_slug': slug,
            **metadata,
        })
        if event is not None and title_date_match(event, event_date):
            return event, request_log
    return None, request_log


def combined_event_text(event: dict[str, Any]) -> str:
    event_fields = ['title', 'subtitle', 'description', 'resolutionSource', 'slug']
    market_fields = [
        'question', 'description', 'resolutionSource', 'slug',
        'groupItemTitle', 'groupItemThreshold',
    ]
    parts = [scalar_text(event.get(field)) for field in event_fields]
    for market in event.get('markets') or []:
        parts.extend(scalar_text(market.get(field)) for field in market_fields)
    return ' '.join(part for part in parts if part).lower()


def source_evidence(event: dict[str, Any]) -> dict[str, bool]:
    text = combined_event_text(event)
    return {
        'hko_evidence': any(marker in text for marker in [
            'hong kong observatory', 'hko.gov.hk', 'weather.gov.hk',
        ]),
        'daily_extract_evidence': 'daily extract' in text,
        'absolute_daily_max_evidence': any(marker in text for marker in [
            'absolute daily maximum', 'absolute daily max',
        ]),
        'one_decimal_evidence': bool(re.search(
            r'one\s+decimal|1\s+decimal|0\.1\s*°?\s*c|9\.1\s*°?\s*c',
            text,
            flags=re.IGNORECASE,
        )),
    }


def label_payload(event_type: str, threshold: float, source: str, text: str) -> dict[str, Any]:
    if event_type == 'lower':
        return {
            'parse_ok': True,
            'event_type': event_type,
            'label_value_c': threshold,
            'lower_bound_c': None,
            'upper_bound_c': threshold + 1.0,
            'canonical_label': f'{threshold:g}°C or below',
            'label_source': source,
            'label_text': text,
        }
    if event_type == 'upper':
        return {
            'parse_ok': True,
            'event_type': event_type,
            'label_value_c': threshold,
            'lower_bound_c': threshold,
            'upper_bound_c': None,
            'canonical_label': f'{threshold:g}°C or higher',
            'label_source': source,
            'label_text': text,
        }
    return {
        'parse_ok': True,
        'event_type': event_type,
        'label_value_c': threshold,
        'lower_bound_c': threshold,
        'upper_bound_c': threshold + 1.0,
        'canonical_label': f'{threshold:g}°C',
        'label_source': source,
        'label_text': text,
    }


def parse_contract_label(market: dict[str, Any]) -> dict[str, Any]:
    candidates = [
        ('groupItemTitle', scalar_text(market.get('groupItemTitle'))),
        ('question', scalar_text(market.get('question'))),
        ('slug', scalar_text(market.get('slug')).replace('-', ' ')),
    ]
    number = r'(-?\d+(?:\.\d+)?)'
    lower_patterns = [
        re.compile(number + r'\s*°?\s*c?\s*(?:or\s*)?(?:below|lower|less|under)', re.I),
        re.compile(r'(?:below|lower|less|under)\s*(?:than|or)?\s*' + number + r'\s*°?\s*c?', re.I),
    ]
    upper_patterns = [
        re.compile(number + r'\s*°?\s*c?\s*(?:or\s*)?(?:higher|above|more|over)', re.I),
        re.compile(r'(?:higher|above|more|over)\s*(?:than|or)?\s*' + number + r'\s*°?\s*c?', re.I),
    ]
    for source, text in candidates:
        if not text:
            continue
        for pattern in lower_patterns:
            match = pattern.search(text)
            if match:
                return label_payload('lower', float(match.group(1)), source, text)
        for pattern in upper_patterns:
            match = pattern.search(text)
            if match:
                return label_payload('upper', float(match.group(1)), source, text)

    exact_patterns = [
        re.compile(r'^\s*' + number + r'\s*°?\s*c?\s*$', re.I),
        re.compile(r'(?:be|at|equal(?:\s+to)?)\s*' + number + r'\s*°?\s*c\b', re.I),
        re.compile(r'\b' + number + r'\s*c\s*$', re.I),
    ]
    for source, text in candidates:
        if not text:
            continue
        for pattern in exact_patterns:
            match = pattern.search(text)
            if match:
                return label_payload('interior', float(match.group(1)), source, text)
    return {
        'parse_ok': False,
        'event_type': '',
        'label_value_c': None,
        'lower_bound_c': None,
        'upper_bound_c': None,
        'canonical_label': '',
        'label_source': '',
        'label_text': '',
    }


def extract_tokens(market: dict[str, Any]) -> dict[str, Any]:
    outcomes = [str(value) for value in parse_jsonish_list(market.get('outcomes'))]
    token_ids = [str(value) for value in parse_jsonish_list(market.get('clobTokenIds'))]
    yes_token_id = ''
    no_token_id = ''
    for index, outcome in enumerate(outcomes):
        if index >= len(token_ids):
            break
        normalised = outcome.strip().lower()
        if normalised == 'yes':
            yes_token_id = token_ids[index]
        elif normalised == 'no':
            no_token_id = token_ids[index]
    fallback = False
    if len(token_ids) == 2 and not yes_token_id and not no_token_id:
        yes_token_id, no_token_id = token_ids
        fallback = True
    return {
        'outcomes_json': json.dumps(outcomes, ensure_ascii=False),
        'clob_token_ids_json': json.dumps(token_ids, ensure_ascii=False),
        'yes_token_id': yes_token_id,
        'no_token_id': no_token_id,
        'token_mapping_fallback': fallback,
    }


def validate_book(rows: list[dict[str, Any]]) -> dict[str, Any]:
    parseable = [row for row in rows if bool(row['parse_ok'])]
    lower = [row for row in parseable if row['event_type'] == 'lower']
    upper = [row for row in parseable if row['event_type'] == 'upper']
    interiors = sorted(
        [row for row in parseable if row['event_type'] == 'interior'],
        key=lambda row: float(row['lower_bound_c']),
    )
    unique_market_ids = len({row['market_id'] for row in rows}) == len(rows)
    unique_labels = len({row['canonical_label'] for row in parseable}) == len(parseable)
    contiguous = len(lower) == 1 and len(upper) == 1 and len(interiors) >= 1
    if contiguous:
        contiguous = abs(float(lower[0]['upper_bound_c']) - float(interiors[0]['lower_bound_c'])) < 1e-9
    if contiguous:
        contiguous = all(
            abs(float(current['upper_bound_c']) - float(following['lower_bound_c'])) < 1e-9
            for current, following in zip(interiors, interiors[1:])
        )
    if contiguous:
        contiguous = abs(float(interiors[-1]['upper_bound_c']) - float(upper[0]['lower_bound_c'])) < 1e-9
    return {
        'n_markets': len(rows),
        'n_parseable': len(parseable),
        'n_lower': len(lower),
        'n_interior': len(interiors),
        'n_upper': len(upper),
        'unique_market_ids': unique_market_ids,
        'unique_labels': unique_labels,
        'partition_contiguous': contiguous,
        'book_complete_11': len(rows) == 11,
        'partition_valid': (
            len(rows) == 11
            and len(parseable) == 11
            and len(lower) == 1
            and len(interiors) == 9
            and len(upper) == 1
            and unique_market_ids
            and unique_labels
            and contiguous
        ),
        'yes_tokens_complete': len(rows) == 11 and all(bool(row['yes_token_id']) for row in rows),
        'no_tokens_complete': len(rows) == 11 and all(bool(row['no_token_id']) for row in rows),
    }


def safe_int(value: Any) -> int:
    return 0 if pd.isna(value) else int(value)


def safe_bool(value: Any) -> bool:
    return False if pd.isna(value) else bool(value)


def sha256_file(path: Path) -> str:
    digest = hashlib.sha256()
    with path.open('rb') as handle:
        for chunk in iter(lambda: handle.read(1024 * 1024), b''):
            digest.update(chunk)
    return digest.hexdigest()


event_rows: list[dict[str, Any]] = []
contract_rows: list[dict[str, Any]] = []
issue_rows: list[dict[str, Any]] = []
request_rows: list[dict[str, Any]] = []
raw_index_rows: list[dict[str, Any]] = []

for event_date in MONTH_DATES:
    event, logs = discover_event(event_date)
    request_rows.extend(logs)
    if event is None:
        event_rows.append({
            'event_date': event_date.isoformat(),
            'event_found': False,
            'event_id': '',
            'event_slug': '',
            'event_title': '',
            'event_url': '',
            'audit_status': 'FAIL',
            'issue_codes': 'EVENT_NOT_FOUND',
        })
        issue_rows.append({
            'event_date': event_date.isoformat(),
            'event_slug': '',
            'market_id': '',
            'issue_level': 'date',
            'issue_code': 'EVENT_NOT_FOUND',
            'detail': 'No exact June Hong Kong temperature event was found.',
        })
        continue

    event_slug = scalar_text(event.get('slug'))
    raw_path = RAW_DIR / f'{event_date.isoformat()}__{event_slug}.json'
    raw_path.write_text(json.dumps(event, indent=2, ensure_ascii=False), encoding='utf-8')
    raw_index_rows.append({
        'event_date': event_date.isoformat(),
        'event_slug': event_slug,
        'raw_path': str(raw_path.relative_to(REPO_ROOT)),
        'sha256': sha256_file(raw_path),
    })

    evidence = source_evidence(event)
    markets = event.get('markets') or []
    date_contract_rows: list[dict[str, Any]] = []

    for market in markets:
        parsed = parse_contract_label(market)
        tokens = extract_tokens(market)
        row = {
            'event_date': event_date.isoformat(),
            'event_id': scalar_text(event.get('id')),
            'event_slug': event_slug,
            'event_title': scalar_text(event.get('title')),
            'event_url': f'https://polymarket.com/event/{event_slug}',
            'market_id': scalar_text(market.get('id')),
            'condition_id': scalar_text(market.get('conditionId')),
            'market_slug': scalar_text(market.get('slug')),
            'question': scalar_text(market.get('question')),
            'group_item_title': scalar_text(market.get('groupItemTitle')),
            'group_item_threshold_raw': scalar_text(market.get('groupItemThreshold')),
            'resolution_source': scalar_text(market.get('resolutionSource')),
            'active': market.get('active'),
            'closed': market.get('closed'),
            **evidence,
            **parsed,
            **tokens,
        }
        reasons: list[str] = []
        if not evidence['hko_evidence']:
            reasons.append('HKO_SOURCE_UNVERIFIED')
        if not evidence['daily_extract_evidence']:
            reasons.append('DAILY_EXTRACT_UNVERIFIED')
        if not evidence['absolute_daily_max_evidence']:
            reasons.append('ABSOLUTE_DAILY_MAX_UNVERIFIED')
        if not evidence['one_decimal_evidence']:
            reasons.append('ONE_DECIMAL_UNVERIFIED')
        if not parsed['parse_ok']:
            reasons.append('LABEL_UNPARSEABLE')
        if not tokens['yes_token_id']:
            reasons.append('YES_TOKEN_MISSING')
        if not tokens['no_token_id']:
            reasons.append('NO_TOKEN_MISSING')
        row['exclusion_reasons'] = ';'.join(reasons)
        date_contract_rows.append(row)
        contract_rows.append(row)
        for reason in reasons:
            issue_rows.append({
                'event_date': event_date.isoformat(),
                'event_slug': event_slug,
                'market_id': row['market_id'],
                'issue_level': 'contract',
                'issue_code': reason,
                'detail': row['question'] or row['group_item_title'],
            })

    checks = validate_book(date_contract_rows)
    date_issues: list[str] = []
    if not evidence['hko_evidence']:
        date_issues.append('HKO_SOURCE_UNVERIFIED')
    if not evidence['daily_extract_evidence']:
        date_issues.append('DAILY_EXTRACT_UNVERIFIED')
    if not evidence['absolute_daily_max_evidence']:
        date_issues.append('ABSOLUTE_DAILY_MAX_UNVERIFIED')
    if not evidence['one_decimal_evidence']:
        date_issues.append('ONE_DECIMAL_UNVERIFIED')
    if checks['n_markets'] != 11:
        date_issues.append(f"MARKET_COUNT_{checks['n_markets']}")
    if checks['n_parseable'] != checks['n_markets']:
        date_issues.append(f"UNPARSEABLE_{checks['n_markets'] - checks['n_parseable']}")
    if not checks['partition_valid']:
        date_issues.append('PARTITION_INVALID')
    if not checks['yes_tokens_complete']:
        date_issues.append('YES_TOKENS_INCOMPLETE')
    if not checks['no_tokens_complete']:
        date_issues.append('NO_TOKENS_INCOMPLETE')
    for issue in date_issues:
        issue_rows.append({
            'event_date': event_date.isoformat(),
            'event_slug': event_slug,
            'market_id': '',
            'issue_level': 'date',
            'issue_code': issue,
            'detail': 'See date-level and contract-level audit tables.',
        })

    event_rows.append({
        'event_date': event_date.isoformat(),
        'event_found': True,
        'event_id': scalar_text(event.get('id')),
        'event_slug': event_slug,
        'event_title': scalar_text(event.get('title')),
        'event_url': f'https://polymarket.com/event/{event_slug}',
        'event_active': event.get('active'),
        'event_closed': event.get('closed'),
        'event_start_date': scalar_text(event.get('startDate')),
        'event_end_date': scalar_text(event.get('endDate')),
        **evidence,
        **checks,
        'audit_status': 'PASS' if not date_issues else 'REVIEW',
        'issue_codes': ';'.join(date_issues),
    })
    time.sleep(REQUEST_SLEEP_SECONDS)


event_df = pd.DataFrame(event_rows).sort_values('event_date').reset_index(drop=True)
contract_df = pd.DataFrame(contract_rows)
if not contract_df.empty:
    contract_df = contract_df.sort_values(
        ['event_date', 'event_type', 'label_value_c', 'market_id'],
        na_position='last',
    ).reset_index(drop=True)
issue_df = pd.DataFrame(issue_rows, columns=['event_date', 'event_slug', 'market_id', 'issue_level', 'issue_code', 'detail'])
request_df = pd.DataFrame(request_rows, columns=['event_date', 'method', 'candidate_slug', 'request_url', 'http_status', 'attempt', 'error'])
raw_index_df = pd.DataFrame(raw_index_rows, columns=['event_date', 'event_slug', 'raw_path', 'sha256'])

if len(event_df) != 30:
    raise AssertionError(f'Expected 30 date-level rows, found {len(event_df)}')
if event_df['event_date'].duplicated().any():
    raise AssertionError('Duplicate June date rows found.')
if not contract_df.empty and contract_df.duplicated(['event_date', 'market_id']).any():
    raise AssertionError('Duplicate date-market keys found.')

paths = {
    'events': OUT_DIR / '18n_june_2026_event_audit.csv',
    'contracts': OUT_DIR / '18n_june_2026_contract_audit.csv',
    'issues': OUT_DIR / '18n_june_2026_contract_issues.csv',
    'requests': OUT_DIR / '18n_june_2026_api_request_log.csv',
    'raw_index': OUT_DIR / '18n_june_2026_raw_event_index.csv',
}
event_df.to_csv(paths['events'], index=False)
contract_df.to_csv(paths['contracts'], index=False)
issue_df.to_csv(paths['issues'], index=False)
request_df.to_csv(paths['requests'], index=False)
raw_index_df.to_csv(paths['raw_index'], index=False)

pass_dates = event_df.loc[event_df['audit_status'].eq('PASS'), 'event_date'].tolist()
review_dates = event_df.loc[event_df['audit_status'].eq('REVIEW'), 'event_date'].tolist()
fail_dates = event_df.loc[event_df['audit_status'].eq('FAIL'), 'event_date'].tolist()
verdict = 'PASS' if len(pass_dates) == 30 else ('USABLE_WITH_LIMITATIONS' if pass_dates else 'NEEDS_CORRECTION')

summary = {
    'step': STEP,
    'generated_at_utc': datetime.now(timezone.utc).isoformat(),
    'verdict': verdict,
    'requested_dates': 30,
    'events_found': int(event_df['event_found'].fillna(False).sum()),
    'pass_dates': len(pass_dates),
    'review_dates': len(review_dates),
    'fail_dates': len(fail_dates),
    'pass_date_list': pass_dates,
    'review_date_list': review_dates,
    'fail_date_list': fail_dates,
    'contract_rows_retrieved': int(len(contract_df)),
    'maximum_possible_contract_rows': 330,
    'pass_contract_rows': int(contract_df['event_date'].isin(pass_dates).sum()) if not contract_df.empty else 0,
    'issue_rows': int(len(issue_df)),
    'interpretation': 'Only PASS dates may enter the June outcome, market-history and weather-path extensions.',
}
summary_path = OUT_DIR / '18n_june_2026_audit_summary.json'
summary_path.write_text(json.dumps(summary, indent=2, ensure_ascii=False), encoding='utf-8')

environment = {
    'generated_at_utc': datetime.now(timezone.utc).isoformat(),
    'python': sys.version,
    'platform': platform.platform(),
    'pandas': pd.__version__,
    'requests': requests.__version__,
    'polymarket_gamma_base_url': BASE_URL,
    'primary_endpoint': f'{BASE_URL}/events/slug/{{slug}}',
    'fallback_endpoint': f'{BASE_URL}/public-search',
}
(OUT_DIR / '18n_june_2026_environment.json').write_text(
    json.dumps(environment, indent=2, ensure_ascii=False), encoding='utf-8'
)

report_lines = [
    '# 18n June 2026 Hong Kong contract-event audit',
    '',
    f"Generated at UTC: `{summary['generated_at_utc']}`",
    '',
    '## Overall judgement',
    '',
    f'**{verdict}**',
    '',
    '## Sample flow',
    '',
    f"- Requested June dates: {summary['requested_dates']}",
    f"- Events found: {summary['events_found']}",
    f"- PASS dates: {summary['pass_dates']}",
    f"- REVIEW dates: {summary['review_dates']}",
    f"- FAIL dates: {summary['fail_dates']}",
    f"- Contract rows retrieved: {summary['contract_rows_retrieved']}",
    f"- Contract rows belonging to PASS dates: {summary['pass_contract_rows']}",
    f"- Issue rows: {summary['issue_rows']}",
    '',
    '## Date-level audit',
    '',
    '| Date | Found | Markets | Lower | Interior | Upper | Partition | Tokens | Status | Issues |',
    '|---|---:|---:|---:|---:|---:|---:|---:|---|---|',
]
for _, row in event_df.iterrows():
    report_lines.append(
        '| {date} | {found} | {markets} | {lower} | {interior} | {upper} | {partition} | {tokens} | {status} | {issues} |'.format(
            date=row['event_date'],
            found=safe_bool(row.get('event_found', False)),
            markets=safe_int(row.get('n_markets', 0)),
            lower=safe_int(row.get('n_lower', 0)),
            interior=safe_int(row.get('n_interior', 0)),
            upper=safe_int(row.get('n_upper', 0)),
            partition=safe_bool(row.get('partition_valid', False)),
            tokens=safe_bool(row.get('yes_tokens_complete', False)) and safe_bool(row.get('no_tokens_complete', False)),
            status=row.get('audit_status', 'FAIL'),
            issues=row.get('issue_codes', '') or '',
        )
    )
report_lines.extend([
    '',
    '## Acceptance rule',
    '',
    'A date passes only when an exact June Hong Kong event is found; the event text evidences the HKO Daily Extract, Absolute Daily Maximum Temperature and one-decimal settlement convention; exactly eleven labels are parseable; the event sets comprise one lower endpoint, nine contiguous one-degree interior bins and one upper tail; and every contract has YES and NO CLOB token identifiers.',
    '',
    '## Downstream use',
    '',
    'Only PASS dates may proceed to the June realised-outcome, market-history and deterministic-weather extensions. REVIEW dates require manual source inspection. FAIL dates remain excluded.',
])
report_path = REPORT_DIR / '18n_june_2026_contract_event_audit_report.md'
report_path.write_text('\n'.join(report_lines) + '\n', encoding='utf-8')

manifest_rows: list[dict[str, Any]] = []
for root in (RAW_DIR, OUT_DIR, REPORT_DIR):
    for path in sorted(root.rglob('*')):
        if not path.is_file() or path.name == '18n_june_2026_sha256_manifest.csv':
            continue
        manifest_rows.append({
            'path': str(path.relative_to(REPO_ROOT)),
            'size_bytes': path.stat().st_size,
            'sha256': sha256_file(path),
        })
manifest_path = OUT_DIR / '18n_june_2026_sha256_manifest.csv'
pd.DataFrame(manifest_rows).to_csv(manifest_path, index=False)

print(json.dumps(summary, indent=2, ensure_ascii=False))
display_columns = [
    'event_date', 'event_slug', 'n_markets', 'n_lower', 'n_interior', 'n_upper',
    'partition_valid', 'yes_tokens_complete', 'no_tokens_complete',
    'audit_status', 'issue_codes',
]
display(event_df.reindex(columns=display_columns))
non_pass = event_df.loc[~event_df['audit_status'].eq('PASS'), display_columns]
if non_pass.empty:
    print('All 30 June dates passed the automated audit.')
else:
    print('Dates requiring review or correction:')
    display(non_pass)
if not issue_df.empty:
    display(
        issue_df.groupby(['issue_level', 'issue_code'], dropna=False)
        .size().rename('count').reset_index()
        .sort_values(['issue_level', 'count', 'issue_code'], ascending=[True, False, True])
    )

{
  "step": "18n",
  "generated_at_utc": "2026-07-21T01:45:15.333065+00:00",
  "verdict": "PASS",
  "requested_dates": 30,
  "events_found": 30,
  "pass_dates": 30,
  "review_dates": 0,
  "fail_dates": 0,
  "pass_date_list": [
    "2026-06-01",
    "2026-06-02",
    "2026-06-03",
    "2026-06-04",
    "2026-06-05",
    "2026-06-06",
    "2026-06-07",
    "2026-06-08",
    "2026-06-09",
    "2026-06-10",
    "2026-06-11",
    "2026-06-12",
    "2026-06-13",
    "2026-06-14",
    "2026-06-15",
    "2026-06-16",
    "2026-06-17",
    "2026-06-18",
    "2026-06-19",
    "2026-06-20",
    "2026-06-21",
    "2026-06-22",
    "2026-06-23",
    "2026-06-24",
    "2026-06-25",
    "2026-06-26",
    "2026-06-27",
    "2026-06-28",
    "2026-06-29",
    "2026-06-30"
  ],
  "review_date_list": [],
  "fail_date_list": [],
  "contract_rows_retrieved": 330,
  "maximum_possible_contract_rows": 330,
  "pass_contract_rows": 330,
  "issue_rows": 0,
  "interpretation": "Only PASS dates may enter the June 

,event_date,event_slug,n_markets,n_lower,n_interior,n_upper,partition_valid,yes_tokens_complete,no_tokens_complete,audit_status,issue_codes
0,2026-06-01,highest-temperature-in-hong-kong-on-june-1-2026,11,1,9,1,True,True,True,PASS,
1,2026-06-02,highest-temperature-in-hong-kong-on-june-2-2026,11,1,9,1,True,True,True,PASS,
2,2026-06-03,highest-temperature-in-hong-kong-on-june-3-2026,11,1,9,1,True,True,True,PASS,
3,2026-06-04,highest-temperature-in-hong-kong-on-june-4-2026,11,1,9,1,True,True,True,PASS,
4,2026-06-05,highest-temperature-in-hong-kong-on-june-5-2026,11,1,9,1,True,True,True,PASS,
5,2026-06-06,highest-temperature-in-hong-kong-on-june-6-2026,11,1,9,1,True,True,True,PASS,
6,2026-06-07,highest-temperature-in-hong-kong-on-june-7-2026,11,1,9,1,True,True,True,PASS,
7,2026-06-08,highest-temperature-in-hong-kong-on-june-8-2026,11,1,9,1,True,True,True,PASS,
8,2026-06-09,highest-temperature-in-hong-kong-on-june-9-2026,11,1,9,1,True,True,True,PASS,
9,2026-06-10,highest-temperature-in-hong-kong-on-june-10-2026,11,1,9,1,True,True,True,PASS,


All 30 June dates passed the automated audit.
